# Lockbox State Extraction (per-view models)

Builds on:
- **three per-view DLC projects** under `…/perview/lockbox_{top,side,front}-htcv-*`, each predicts only the
  keypoints its camera sees, so predictions come from three folders, not one pooled project.
- `01_lockbox_calibration.ipynb` to the **same** shared aniposelib `CameraGroup` (PnP poses to world = lockbox frame)
- `state_schema.json` to mechanism schema (axes, origins, ranges)

Pipeline:
1. Load schema + the per-view DLC predictions (`.h5` per project) into a `(cams, frames, kps, 2)` tensor.
   Keypoints = **union** across views (each view contributes its own subset).
2. Triangulate with the shared calibration, N-view wherever >=2 cams see a point. Annotate 3/2/1-view provenance.
3. Single-view fallback (flagged) + short-gap fill.
4. Continuous state per mechanism via the schema axes (`angle_around_axis` / `project_on_axis` / `identity_3d`).
5. **Discrete lockbox state machine**: `start to lever1 pivoted to slider1 slid to ball removed to cover slid to end`.
6. Export a **3D reconstruction** video (triangulated keypoints in lockbox space, colored by 1/2/3-view provenance).
7. Save `lockbox_state.csv` + `lockbox_state.pkl` (continuous states, discrete stage, raw 2D, 3D, provenance).

> End goal (next notebook, not built here): predict the discrete lockbox stage three ways as a sanity check
> from the raw 2D mechanism predictions, from the reconstructed 3D, and from the **mouse** predictions. This
> notebook saves everything those predictors need as ground truth + features.

## 1. Imports & paths

In [ ]:
import os, json, pickle, warnings, collections
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

import cv2
from aniposelib.cameras import CameraGroup

warnings.filterwarnings('ignore')

# Paths
PERVIEW_ROOT = Path('../data/lockbox_dlc/scene1/perview').resolve()
SCHEMA_PATH  = Path('../data/lockbox_dlc/scene1/state_schema.json').resolve()
LOCKBOX_DIR  = Path('../data/lockbox_calibration/scene1').resolve()
OUT_DIR      = Path('../data/lockbox_state/scene1').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCORER = 'htcv'
CONF   = 0.5          # likelihood cutoff
FPS    = 30

## 2. Load the lockbox schema

In [2]:
with open(SCHEMA_PATH) as f:
    LOCKBOX_SCHEMA = json.load(f)
assert LOCKBOX_SCHEMA.get('coord_frame') == 'lockbox', "coord_frame != 'lockbox'"

MECHANISMS, STATE_RANGE, RANGE_BOX = {}, {}, {}
MOVING_KPS, STATIC_KPS = [], []
for m in LOCKBOX_SCHEMA['mechanisms']:
    name, t = m['name'], m['type']
    moving  = m['state_keypoint']
    refs    = list(m.get('reference_keypoints', []))
    MOVING_KPS.append(moving); STATIC_KPS.extend(refs)
    e = dict(type=t, moving=moving, refs=refs, extraction=m.get('state_extraction'))
    if t in ('revolute', 'prismatic'):
        d = np.asarray(m['axis_direction_lockbox'], float)
        e['axis_origin'] = np.asarray(m['axis_origin_lockbox'], float)
        e['axis_dir']    = d / np.linalg.norm(d)
        if t == 'revolute':
            e['unit'] = 'rad'
            STATE_RANGE[name] = tuple(m['range_rad'])
            e['extraction'] = e['extraction'] or 'angle_around_axis'
            if 'rest_direction_lockbox' in m:
                r = np.asarray(m['rest_direction_lockbox'], float)
                e['rest_dir'] = r / np.linalg.norm(r)
            e['preferred_view'] = m.get('preferred_view')
        else:
            e['unit'] = 'mm'
            STATE_RANGE[name] = tuple(m['range_mm'])
            e['extraction'] = e['extraction'] or 'project_on_axis'
    elif t == 'free':
        e['unit'] = 'mm'
        e['rest_pos'] = np.asarray(m['rest_position_lockbox'], float)
        e['extraction'] = e['extraction'] or 'distance_from_rest'
        e['removed_radius_mm'] = float(m.get('removed_radius_mm', 20.0))
        RANGE_BOX[name] = m.get('range_mm_box')
    else:
        raise ValueError(f'{name}: unknown type {t!r}')
    MECHANISMS[name] = e

MOVING_KPS = sorted(set(MOVING_KPS)); STATIC_KPS = sorted(set(STATIC_KPS))
BODYPARTS  = sorted(set(MOVING_KPS) | set(STATIC_KPS))
print(f'mechanisms: {list(MECHANISMS)}')
print(f'bodyparts ({len(BODYPARTS)}): {BODYPARTS}')

mechanisms: ['lever1', 'slider1', 'ball1', 'cover1']
bodyparts (8): ['ball1_center', 'cover_guidance_left', 'cover_guidance_right', 'cover_marker', 'lever1_pivot', 'lever1_push_end', 'slider1_knob', 'slider1_stabilizer']


## 2b. Lockbox state machine

The box is a sequential puzzle: each step engages one mechanism, which enables the next. Each step is
detected by thresholding that mechanism's continuous state. Onsets are enforced **in order** and debounced.

```
start to lever1 pivoted to slider1 slid to ball removed to cover slid to end
```

Tune the `ENGAGE` thresholds after seeing the time series. (If the lever angle comes out inverted
rest ≈ 0 instead of ≈ π/2, flip its rule to `('above', …)` or flip `axis_direction` in the schema.)

In [3]:
ENGAGE = {
    'lever1':  ('above', np.deg2rad(45)),                     # arcsin gives [0, π/2]; 45° = halfway flipped
    'slider1': ('above', 5.0),
    'ball1':   ('above', MECHANISMS['ball1']['removed_radius_mm']),
    'cover1':  ('above', 10.0),
}
STAGE_ORDER = ['lever1', 'slider1', 'ball1', 'cover1']
STAGE_NAMES = ['start', 'lever1_pivoted', 'slider1_slid', 'ball_removed', 'cover_slid']
MIN_PERSIST = 5
print('stages:', ' -> '.join(STAGE_NAMES))
for n in STAGE_ORDER:
    op, thr = ENGAGE[n]
    print(f'  {n:<8} {op:<10} {thr:.3f}')

stages: start -> lever1_pivoted -> slider1_slid -> ball_removed -> cover_slid
  lever1   above      0.785
  slider1  above      5.000
  ball1    above      30.000
  cover1   above      10.000


## 3. Load calibration & per-view predictions

`CameraGroup` is the single shared calibration. Each per-view project is found by glob, and its
`analyze_videos` `.h5` is matched by the `lockbox_<view>` tag in the filename. `valid_kps` is the **union**
of bodyparts across views (per-view models each predict only their labeled subset).

In [ ]:
cgroup     = CameraGroup.load(str(LOCKBOX_DIR / 'lockbox_calibration.toml'))
view_order = [c.get_name() for c in cgroup.cameras]
n_cams     = len(view_order)

def discover_view(view):
    cfgp = next(iter(sorted(PERVIEW_ROOT.glob(f'lockbox_{view}-{SCORER}-*/config.yaml'))), None)
    if cfgp is None:
        return dict(cfg=None, video=None, h5=None)
    cfg  = yaml.safe_load(open(cfgp))
    vids = list(cfg.get('video_sets', {}).keys())
    video = Path(vids[0]) if vids else None
    cands = []
    for d in [cfgp.parent / 'videos', (video.parent if video else None)]:
        if d and Path(d).exists():
            cands += [h for h in Path(d).glob('*.h5')
                      if f'lockbox_{view}' in h.name and 'filtered' not in h.name.lower()]
    return dict(cfg=cfgp, video=video, h5=(sorted(cands)[-1] if cands else None))

VIEW_INFO      = {v: discover_view(v) for v in view_order}
H5_FOR_VIEW    = {v: VIEW_INFO[v]['h5']    for v in view_order}
VIDEO_FOR_VIEW = {v: VIEW_INFO[v]['video'] for v in view_order}
print('Calibration cameras:', view_order)
for v in view_order:
    info = VIEW_INFO[v]
    print(f'  {v:<6} proj={info["cfg"].parent.name if info["cfg"] else "MISSING":<32} '
          f'h5={info["h5"].name if info["h5"] else "run analyze_videos for this view"}')

Calibration cameras: ['top', 'side', 'front']
  top    proj=lockbox_top-htcv-2026-05-25      h5=2021-06-18_07-28-45_segment1_mouse291_combined_top-down-viewDLC_Resnet50_lockbox_topMay25shuffle1_snapshot_best-120.h5
  side   proj=lockbox_side-htcv-2026-05-25     h5=2021-06-18_07-28-45_segment1_mouse291_combined_side-viewDLC_Resnet50_lockbox_sideMay25shuffle1_snapshot_best-130.h5
  front  proj=lockbox_front-htcv-2026-05-25    h5=2021-06-18_07-28-45_segment1_mouse291_combined_front-viewDLC_Resnet50_lockbox_frontMay25shuffle1_snapshot_best-120.h5


In [5]:
def load_dlc_h5(path):
    df = pd.read_hdf(path)
    while df.columns.nlevels > 2:          # (scorer[, individuals], bodyparts, coords) -> (bodyparts, coords)
        df = df.droplevel(0, axis=1)
    return df

view_predictions = {}
for v in view_order:
    if H5_FOR_VIEW[v] is None:
        raise FileNotFoundError(f'No .h5 for view {v} under {PERVIEW_ROOT}')
    view_predictions[v] = load_dlc_h5(H5_FOR_VIEW[v])

n_frames  = min(len(df) for df in view_predictions.values())
present   = set().union(*[set(df.columns.get_level_values(0)) for df in view_predictions.values()])  # UNION
valid_kps = [bp for bp in BODYPARTS if bp in present]
n_kps     = len(valid_kps)

print(f'Frames: {n_frames}   Keypoints (union): {n_kps}/{len(BODYPARTS)}')
for v in view_order:
    vb = [bp for bp in valid_kps if bp in set(view_predictions[v].columns.get_level_values(0))]
    print(f'  {v:<6} predicts {len(vb):>2}: {vb}')

Frames: 8190   Keypoints (union): 8/8
  top    predicts  7: ['ball1_center', 'cover_guidance_left', 'cover_guidance_right', 'cover_marker', 'lever1_push_end', 'slider1_knob', 'slider1_stabilizer']
  side   predicts  8: ['ball1_center', 'cover_guidance_left', 'cover_guidance_right', 'cover_marker', 'lever1_pivot', 'lever1_push_end', 'slider1_knob', 'slider1_stabilizer']
  front  predicts  6: ['ball1_center', 'cover_guidance_right', 'lever1_pivot', 'lever1_push_end', 'slider1_knob', 'slider1_stabilizer']


## 4. Merge 2D predictions

`(3 cams, n_frames, n_kps, 2)`, NaN where likelihood < `CONF`. Per-view bodyparts differ, so the loop
simply skips any keypoint a given view doesn't predict (it stays NaN for that camera).

In [6]:
points_2d   = np.full((n_cams, n_frames, n_kps, 2), np.nan, dtype=np.float64)
likelihoods = np.zeros((n_cams, n_frames, n_kps), dtype=np.float32)

for c, view in enumerate(view_order):
    df   = view_predictions[view]
    cols = set(df.columns.get_level_values(0))
    for k, bp in enumerate(valid_kps):
        if bp not in cols: continue                       # this view doesn't see this keypoint
        x = df[bp]['x'].values[:n_frames]
        y = df[bp]['y'].values[:n_frames]
        l = df[bp]['likelihood'].values[:n_frames]
        m = l > CONF
        points_2d[c, m, k, 0] = x[m]
        points_2d[c, m, k, 1] = y[m]
        likelihoods[c, :, k]  = l

views_per_kp = (~np.isnan(points_2d[..., 0])).sum(axis=0)
print(f'points_2d shape: {points_2d.shape}')
print(f'Mean #views per kp per frame: {views_per_kp.mean():.2f}')
print(f'Frames with >=1 triangulable kp (>=2 views): {(views_per_kp >= 2).any(axis=1).sum()} / {n_frames}')

points_2d shape: (3, 8190, 8, 2)
Mean #views per kp per frame: 2.06
Frames with >=1 triangulable kp (>=2 views): 8190 / 8190


## 5. Triangulation + provenance (3 / 2 / 1 view)

In [7]:
points_3d    = np.full((n_frames, n_kps, 3), np.nan, dtype=np.float64)
n_views_used = np.zeros((n_frames, n_kps), dtype=np.int8)

for f in range(n_frames):
    fp = points_2d[:, f, :, :]
    n_views_used[f] = (~np.isnan(fp[..., 0])).sum(axis=0)
    mask = n_views_used[f] >= 2
    if not mask.any(): continue
    points_3d[f, mask] = cgroup.triangulate(fp[:, mask, :], undistort=True)

cov = np.isfinite(points_3d).all(axis=-1)
print(f'Overall coverage (>=2 views): {cov.mean()*100:.1f}%\n')
print(f'{"keypoint":<22}{"type":<8}{"cov":>7}   {"3v":>6}{"2v":>6}{"1v":>6}{"0v":>6}')
for k, bp in enumerate(valid_kps):
    tag = 'moving' if bp in MOVING_KPS else 'static'
    nv  = n_views_used[:, k]
    print(f'{bp:<22}{tag:<8}{cov[:,k].mean()*100:6.1f}%   '
          f'{(nv==3).mean()*100:5.1f}%{(nv==2).mean()*100:5.1f}%'
          f'{(nv==1).mean()*100:5.1f}%{(nv==0).mean()*100:5.1f}%')

Overall coverage (>=2 views): 78.0%

keypoint              type        cov       3v    2v    1v    0v
ball1_center          moving    80.5%    63.9% 16.6%  7.1% 12.4%
cover_guidance_left   static    48.7%     0.0% 48.7% 43.4%  7.9%
cover_guidance_right  static    68.8%    27.1% 41.7% 22.2%  9.0%
cover_marker          moving    60.8%     0.0% 60.8% 18.4% 20.7%
lever1_pivot          static    83.1%     0.0% 83.1% 15.2%  1.7%
lever1_push_end       moving    87.4%     0.0% 87.4% 11.3%  1.3%
slider1_knob          moving    96.7%    92.0%  4.6%  1.9%  1.4%
slider1_stabilizer    static    98.4%    94.2%  4.2%  1.1%  0.4%


## 5b. Frame check: triangulation frame == lockbox frame?

`lever1_pivot ≈ lever1.axis_origin` and `ball1_cradle ≈ ball1.rest_pos` (within a few mm) confirms the
calibration world frame matches the schema's lockbox frame. Large Δ to apply a rigid transform first.

In [8]:
def _median_point(bp):
    if bp not in valid_kps: return None
    col = points_3d[:, valid_kps.index(bp), :]
    col = col[np.isfinite(col).all(axis=1)]
    return np.median(col, axis=0) if len(col) else None

frame_checks = [('lever1_pivot', MECHANISMS['lever1']['axis_origin']),
                ('ball1_cradle', MECHANISMS['ball1']['rest_pos'])]
print(f'{"static ref":<18}{"triang. median":<26}{"schema":<26}{"d (mm)":>8}')
for bp, ref in frame_checks:
    mp = _median_point(bp)
    if mp is None:
        print(f'{bp:<18}{"- (needs >=2 views)":<26}{str(np.round(ref,1)):<26}{"":>8}'); continue
    print(f'{bp:<18}{str(np.round(mp,1)):<26}{str(np.round(ref,1)):<26}{np.linalg.norm(mp-ref):8.1f}')

static ref        triang. median            schema                      d (mm)
lever1_pivot      [ 4.3 32.5 27.4]          [ 4.3 32.5 27.4]               0.0
ball1_cradle      - (needs >=2 views)       [ 50.  -22.6   8.2]               


## 6. Single-view fallback (optional, flagged)

A moving keypoint seen by only one cam is projected onto its back-ray and snapped to the point closest to
the mechanism's first static reference. Flagged in `single_view_mask`. Set `ENABLE_SINGLE_VIEW = False` to skip.

In [9]:
ENABLE_SINGLE_VIEW = True
FALLBACK_REF = {m['moving']: (m['refs'][0] if m['refs'] else None) for m in MECHANISMS.values()}

def cam_center(cam):
    R, _ = cv2.Rodrigues(cam.rvec)
    return (-R.T @ cam.tvec.reshape(3, 1)).ravel()

def back_ray(cam, pt2d):
    pt = np.array([pt2d[0], pt2d[1], 1.0])
    R, _ = cv2.Rodrigues(cam.rvec)
    d = R.T @ (np.linalg.inv(cam.matrix) @ pt); d /= np.linalg.norm(d)
    return cam_center(cam), d

def snap_to_anchor(cam, pt2d, anchor_3d):
    C, d = back_ray(cam, pt2d)
    return C + np.dot(anchor_3d - C, d) * d

static_median = {bp: _median_point(bp) for bp in STATIC_KPS if _median_point(bp) is not None}
single_view_mask = np.zeros((n_frames, n_kps), dtype=bool)
if ENABLE_SINGLE_VIEW:
    for f in range(n_frames):
        for k, bp in enumerate(valid_kps):
            if np.isfinite(points_3d[f, k]).all(): continue
            if n_views_used[f, k] != 1:            continue
            ref = FALLBACK_REF.get(bp)
            anchor = static_median.get(ref) if ref in static_median else None
            if anchor is None and ref in valid_kps and np.isfinite(points_3d[f, valid_kps.index(ref)]).all():
                anchor = points_3d[f, valid_kps.index(ref)]
            if anchor is None: continue
            c = int(np.where(~np.isnan(points_2d[:, f, k, 0]))[0][0])
            points_3d[f, k]        = snap_to_anchor(cgroup.cameras[c], points_2d[c, f, k], anchor)
            single_view_mask[f, k] = True
print(f'Single-view back-projected: {single_view_mask.sum()} points')
print(f'Coverage after fallback: {np.isfinite(points_3d).all(axis=-1).mean()*100:.1f}%')

Single-view back-projected: 2590 points
Coverage after fallback: 82.0%


## 7. Fill short gaps (optional)

In [10]:
def fill_short_gaps(arr, max_gap=5):
    out = arr.copy(); n_f, n_k, _ = out.shape
    for k in range(n_k):
        for c in range(3):
            s = out[:, k, c]; valid = np.isfinite(s)
            if valid.sum() < 2: continue
            invalid = ~valid; diff = np.diff(invalid.astype(int))
            starts = np.where(diff == 1)[0] + 1; ends = np.where(diff == -1)[0] + 1
            if invalid[0]:  starts = np.concatenate([[0], starts])
            if invalid[-1]: ends   = np.concatenate([ends, [n_f]])
            for gs, ge in zip(starts, ends):
                if (ge - gs) > max_gap or gs == 0 or ge == n_f: continue
                y0, y1 = s[gs-1], s[ge]
                for fi in range(gs, ge):
                    out[fi, k, c] = y0 + (y1 - y0) * (fi - (gs-1)) / (ge - (gs-1))
    return out

MAX_GAP = 5
before  = np.isfinite(points_3d).all(axis=-1)
points_3d = fill_short_gaps(points_3d, max_gap=MAX_GAP)
interp_mask = np.isfinite(points_3d).all(axis=-1) & ~before
print(f'Interpolated: {interp_mask.sum()} points (max_gap={MAX_GAP})')
print(f'Coverage after gap-filling: {np.isfinite(points_3d).all(axis=-1).mean()*100:.1f}%')

Interpolated: 527 points (max_gap=5)
Coverage after gap-filling: 82.8%


## 8. Continuous state per mechanism (schema axes)

In [ ]:
def _track(bp):  return points_3d[:, valid_kps.index(bp), :]
def _nviews(bp): return n_views_used[:, valid_kps.index(bp)]

def project_on_axis(track, origin, axis):
    return (track - origin) @ axis

def plane_basis(axis):
    W = np.eye(3); ref = W[int(np.argmin(np.abs(W @ axis)))]
    e1 = ref - (ref @ axis) * axis; e1 /= np.linalg.norm(e1)
    return e1, np.cross(axis, e1)

def angle_around_axis(track, origin, axis):
    v = track - origin; e1, e2 = plane_basis(axis)
    ang = np.arctan2(v @ e2, v @ e1)
    ang[~np.isfinite(track).all(axis=1)] = np.nan
    return ang

def lever_angle_top_apparent_length(mech, view_order, cgroup, points_2d, points_3d,
                                    valid_kps, n_frames):
    """
    Top-view-only lever angle via apparent length (in [0, π/2], magnitude only).

    The rotation plane (perpendicular to axis_dir) is edge-on to the top camera,
    so a 2D-angle decomposition is singular. Instead:
      • L_3D  — 3D lever arm length, from triangulated push_end ↔ pivot distance
      • L_2D  — apparent length of L_3D when projected into top view along the
                in-plane axis perpendicular to rest_dir (the visible direction)
      • d_2D  — observed 2D distance from projected pivot to tracked push_end
      • angle = arcsin( clip(d_2D / L_2D, 0, 1) )

    Returns
    -------
    angle      : (n_frames,) float — 0 at rest, π/2 at full flip, NaN where missing.
    detected   : (n_frames,) int8  — 1 if the top view detected the moving kp.
    L_3D, L_2D : floats             — for diagnostics / printing.
    """
    top_view = mech['preferred_view']
    if top_view not in view_order:
        raise ValueError(f"preferred_view {top_view!r} not in {view_order}")
    cam_idx = view_order.index(top_view)
    cam     = cgroup.cameras[cam_idx]

    axis_dir = mech['axis_dir']
    rest_dir = mech.get('rest_dir')
    if rest_dir is None:
        rest_dir, _ = plane_basis(axis_dir)
    perp = np.cross(axis_dir, rest_dir); perp /= np.linalg.norm(perp)

    pivot_3d = mech['axis_origin']
    k = valid_kps.index(mech['moving'])

    # --- d_2D: observed top-view distance from projected pivot to tracked push_end
    pivot_2d = cam.project(pivot_3d.reshape(1, 3))[0]
    pe_2d    = points_2d[cam_idx, :, k, :]
    d_2d     = np.linalg.norm(pe_2d - pivot_2d, axis=1)
    detected = np.isfinite(d_2d).astype(np.int8)
    if detected.sum() < 10:
        return np.full(n_frames, np.nan), detected, np.nan, np.nan

    # --- L_3D: lever arm length from the 3D track of push_end relative to pivot.
    #     The arm is rigid -> distance is constant up to triangulation noise.
    track_3d = points_3d[:, k, :]
    finite   = np.isfinite(track_3d).all(axis=1)
    if finite.sum() >= 10:
        L_3D = float(np.median(np.linalg.norm(track_3d[finite] - pivot_3d, axis=1)))
    else:
        # Fallback: assume the data spans a full flip; calibrate from p99 of d_2D.
        L_3D = None

    # --- L_2D: apparent length in the top view at full flip (in the perp direction).
    if L_3D is not None and np.isfinite(L_3D) and L_3D > 0.5:
        ref_2d = cam.project(np.vstack([pivot_3d, pivot_3d + L_3D * perp]))
        L_2D   = float(np.linalg.norm(ref_2d[1] - ref_2d[0]))
    else:
        L_2D = float(np.nanpercentile(d_2d, 99))
        L_3D = np.nan

    if not np.isfinite(L_2D) or L_2D < 1.0:
        return np.full(n_frames, np.nan), detected, L_3D, L_2D

    ratio = np.clip(d_2d / L_2D, 0.0, 1.0)
    angle = np.arcsin(ratio)
    angle[~np.isfinite(d_2d)] = np.nan
    return angle, detected, L_3D, L_2D

state = {}
for name, m in MECHANISMS.items():
    mv = m['moving']
    if mv not in valid_kps:
        print(f'{name}: moving kp "{mv}" not predicted by any view — skipped'); continue
    nv  = _nviews(mv)
    ext = m['extraction']

    if ext == 'project_on_axis':
        track = _track(mv)
        v = project_on_axis(track, m['axis_origin'], m['axis_dir'])
        state[name] = dict(value=v, nviews=nv, unit=m['unit'], kind='prismatic')

    elif ext == 'angle_around_axis':
        track = _track(mv)
        v = angle_around_axis(track, m['axis_origin'], m['axis_dir'])
        state[name] = dict(value=v, nviews=nv, unit=m['unit'], kind='revolute')

    elif ext in ('angle_relative_top_view', 'angle_top_apparent_length'):
        v, det, L3, L2 = lever_angle_top_apparent_length(
            m, view_order, cgroup, points_2d, points_3d, valid_kps, n_frames
        )
        state[name] = dict(value=v, nviews=det, unit=m['unit'], kind='revolute_top',
                           L_3D=L3, L_2D=L2)

    elif ext in ('identity_3d', 'distance_from_rest'):
        track = _track(mv)
        disp  = track - m['rest_pos']
        state[name] = dict(value=np.linalg.norm(disp, axis=1),
                           xyz=track, disp=disp, nviews=nv, unit=m['unit'], kind='free')
    else:
        raise ValueError(f'{name}: unknown extraction {ext!r}')

print(f'{"mechanism":<10}{"kind":<14}{"cov":>7}   range (pre-filter)')
for name, st in state.items():
    v = st['value']
    rng = f'[{np.nanmin(v):.2f}, {np.nanmax(v):.2f}] {st["unit"]}' if np.isfinite(v).any() else '(empty)'
    extra = ''
    if st['kind'] == 'revolute_top':
        extra = f"   [L_3D={st['L_3D']:.1f} mm, L_2D={st['L_2D']:.1f} px]"
    print(f'{name:<10}{st["kind"]:<14}{np.isfinite(v).mean()*100:6.1f}%   {rng}{extra}')

mechanism kind              cov   range (pre-filter)
lever1    revolute_top    87.6%   [0.10, 1.57] rad   [L_3D=65.6 mm, L_2D=339.7 px]
slider1   prismatic       98.7%   [-56.53, 32.61] mm
ball1     free            81.6%   [17.66, 114.24] mm
cover1    prismatic       80.6%   [-43.04, 22.24] mm


## 9. Sanity-filter via `range_*`

In [12]:
for name, st in state.items():
    if st['kind'] == 'free':
        box = RANGE_BOX.get(name)
        if box is None:
            print(f'{name:<10} free: no range_mm_box -> no filter'); continue
        box = np.asarray(box, float); xyz = st['xyz']
        bad = np.isfinite(xyz).all(axis=1) & ~np.all((xyz >= box[:, 0]) & (xyz <= box[:, 1]), axis=1)
        for key in ('xyz', 'disp'): st[key][bad] = np.nan
        st['value'][bad] = np.nan
        print(f'{name:<10} free: {int(bad.sum())} frames outside box -> NaN')
    else:
        lo_r, hi_r = STATE_RANGE.get(name, (-np.inf, np.inf))
        v   = st['value']
        bad = np.isfinite(v) & ((v < lo_r) | (v > hi_r))
        v[bad] = np.nan
        st['value'] = v
        n_in = np.isfinite(v).sum()
        print(f'{name:<10} outside [{lo_r:.2f}, {hi_r:.2f}] {st["unit"]}: {int(bad.sum())} -> NaN '
              f'(remaining {n_in}/{len(v)} = {n_in/len(v)*100:.1f}%)')

lever1     outside [-1.77, 1.77] rad: 0 -> NaN (remaining 7173/8190 = 87.6%)
slider1    outside [-2.00, 40.00] mm: 59 -> NaN (remaining 8028/8190 = 98.0%)
ball1      free: no range_mm_box -> no filter
cover1     outside [-2.00, 30.00] mm: 13 -> NaN (remaining 6590/8190 = 80.5%)


## 10. Discrete stage timeline

Each mechanism is "engaged" when its state crosses its `ENGAGE` threshold (NaN to not engaged). Onsets are
detected in `STAGE_ORDER` with a `MIN_PERSIST` debounce. Each onset must come at/after the previous one.
`stage` is the cumulative (non-decreasing) progress index, with names in `STAGE_NAMES`.

In [13]:
def engaged_mask(name):
    if name not in state: return np.zeros(n_frames, bool)
    v = state[name]['value']
    op, thr = ENGAGE[name]
    if   op == 'below':     e = (v <  thr)
    elif op == 'above':     e = (v >  thr)
    elif op == 'abs_above': e = (np.abs(v) > thr)
    elif op == 'abs_below': e = (np.abs(v) < thr)
    else: raise ValueError(f'Unknown ENGAGE operator: {op!r}')
    return np.where(np.isfinite(v), e, False)

engaged = {name: engaged_mask(name) for name in STAGE_ORDER}

def first_sustained(b, k, start=0):
    cnt = 0
    for i in range(start, len(b)):
        cnt = cnt + 1 if b[i] else 0
        if cnt >= k: return i - k + 1
    return None

onsets, prev = {}, 0
for name in STAGE_ORDER:
    o = first_sustained(engaged[name], MIN_PERSIST, start=prev)
    onsets[name] = o
    if o is not None: prev = o

stage = np.zeros(n_frames, dtype=int)
for idx, name in enumerate(STAGE_ORDER, start=1):
    if onsets[name] is not None: stage[onsets[name]:] = idx

print('Stage onsets (sequential):')
for idx, name in enumerate(STAGE_ORDER, start=1):
    o = onsets[name]
    when = f'frame {o} ({o/FPS:.1f}s)' if o is not None else 'NOT reached'
    print(f'  {idx}. {STAGE_NAMES[idx]:<16} ({name}) -> {when}')
print('Per-mechanism engagement coverage:')
for name in STAGE_ORDER:
    print(f'  {name:<8} engaged {engaged[name].mean()*100:5.1f}% '
          f'longest run = {int(np.diff(np.concatenate([[0], np.where(np.diff(engaged[name].astype(int))!=0)[0]+1, [n_frames]])).max()) if engaged[name].any() else 0} frames')
print('Frames per stage:')
for i, nm in enumerate(STAGE_NAMES):
    print(f'  {i} {nm:<16} {int((stage==i).sum())}')

Stage onsets (sequential):
  1. lever1_pivoted   (lever1) -> frame 5393 (179.8s)
  2. slider1_slid     (slider1) -> frame 5600 (186.7s)
  3. ball_removed     (ball1) -> frame 6265 (208.8s)
  4. cover_slid       (cover1) -> frame 6343 (211.4s)
Per-mechanism engagement coverage:
  lever1   engaged  23.4% longest run = 5393 frames
  slider1  engaged  31.2% longest run = 5586 frames
  ball1    engaged  22.8% longest run = 6265 frames
  cover1   engaged  22.0% longest run = 2574 frames
Frames per stage:
  0 start            5393
  1 lever1_pivoted   207
  2 slider1_slid     665
  3 ball_removed     78
  4 cover_slid       1847


## 11. QC: state time series + stage strip

Green = 3 views, amber = 2 views, red triangle = 1-view fallback. Dashed purple = engage threshold;
dotted black = detected stage onsets; **solid crimson = ground-truth state change** from the video's
annotation CSV (`<mechanism>_state_A1/_A2`). Bottom strip = the discrete stage. The printed table
compares detected vs. ground-truth onsets per mechanism (Δ in seconds).

In [ ]:
mechs = list(state.keys())
fig, axes = plt.subplots(len(mechs) + 1, 1, figsize=(13, 2.1 * (len(mechs) + 1)), sharex=True)
t = np.arange(n_frames) / FPS
onset_t = [onsets[m] / FPS for m in STAGE_ORDER if onsets.get(m) is not None]
GT_CSV = Path('../data/validation/scene1/'
              '2021-06-18_07-28-45_segment1_mouse291_combined_labels.csv').resolve()
GT_STATE_COL = {'lever1': 'lever_state', 'slider1': 'stick_state',
                'ball1': 'ball_state', 'cover1': 'sliding_door_state'}
gt_onsets = {}
if GT_CSV.exists():
    gt_df = pd.read_csv(GT_CSV)
    for name, base in GT_STATE_COL.items():
        cols = [c for c in (f'{base}_A1', f'{base}_A2') if c in gt_df.columns]
        if not cols:
            continue
        engaged_any = (gt_df[cols].to_numpy() > 0).any(axis=1)   # truth in either channel
        idx = np.flatnonzero(engaged_any[:n_frames])
        if idx.size:
            gt_onsets[name] = idx[0] / FPS
    print('Ground-truth state-change onsets [s]:',
          {k: round(v, 1) for k, v in gt_onsets.items()})
    for name in STAGE_ORDER:
        g, d = gt_onsets.get(name), (onsets.get(name) / FPS if onsets.get(name) is not None else None)
        if g is not None and d is not None:
            print(f'  {name:<8} GT={g:6.1f}s  detected={d:6.1f}s  Δ={d - g:+5.1f}s')
else:
    print(f'[QC] ground-truth CSV not found, skipping GT overlay: {GT_CSV}')

for ax, (name, st) in zip(axes[:-1], state.items()):
    v, nv = st['value'], st['nviews']
    sv = single_view_mask[:, valid_kps.index(MECHANISMS[name]['moving'])]
    ax.plot(t, v, '-', lw=0.6, color='0.8', zorder=1)
    for tier, col, lab in [(3, 'forestgreen', '3v'), (2, 'goldenrod', '2v')]:
        mt = (nv == tier) & np.isfinite(v) & ~sv
        ax.scatter(t[mt], v[mt], s=6, c=col, label=lab, zorder=3)
    ms = sv & np.isfinite(v)
    ax.scatter(t[ms], v[ms], s=12, c='tomato', marker='^', label='1v', zorder=4)
    if name in ENGAGE: ax.axhline(ENGAGE[name][1], ls='--', c='purple', lw=0.8)
    for ot in onset_t: ax.axvline(ot, color='k', ls=':', lw=0.5)
    if name in gt_onsets:
        ax.axvline(gt_onsets[name], color='crimson', ls='-', lw=1.5, zorder=5,
                   label='GT change')
    ax.set_ylabel(f'{("dist" if st["kind"]=="free" else name)}\n[{st["unit"]}]')
    ax.set_title(f'{name} ({st["kind"]})', fontsize=9, loc='left')
    ax.legend(fontsize=7, ncol=4, loc='upper right')

axs = axes[-1]
axs.step(t, stage, where='post', color='navy', lw=1.3)
axs.set_yticks(range(len(STAGE_NAMES))); axs.set_yticklabels(STAGE_NAMES, fontsize=7)
axs.set_ylabel('stage')
for ot in onset_t: axs.axvline(ot, color='k', ls=':', lw=0.5)
for name in STAGE_ORDER:
    if name in gt_onsets:
        axs.axvline(gt_onsets[name], color='crimson', ls='-', lw=1.2, zorder=5)
axs.set_xlabel('time [s]')
plt.tight_layout()
plt.savefig(OUT_DIR / 'lockbox_state_timeseries.png', dpi=110, bbox_inches='tight'); plt.show()

Ground-truth state-change onsets [s]: {'lever1': 179.5, 'slider1': 192.8, 'ball1': 193.2, 'cover1': 211.5}
  lever1   GT= 179.5s  detected= 179.8s  Δ= +0.2s
  slider1  GT= 192.8s  detected= 186.7s  Δ= -6.1s
  ball1    GT= 193.2s  detected= 208.8s  Δ=+15.6s
  cover1   GT= 211.5s  detected= 211.4s  Δ= -0.1s


## 12. 3D reconstruction export

Renders the triangulated lockbox in its own coordinate frame over time, points colored by provenance
(green=3v, amber=2v, red=1-view fallback, purple=interpolated) with mechanism links as a skeleton, the
current stage in the title. Rendered headless (Agg) to an mp4. Slow for the full video, narrow the window
or raise `RENDER_STRIDE` to speed it up.

In [ ]:
out_mp4 = OUT_DIR / 'lockbox_3d_reconstruction.mp4'

# The 3D reconstruction clip is a visualization artifact only; cells 13/14 save
# the CSV / pkl from points_3d + state, not from this video.

def _has_video(p):
    return p.exists() and p.stat().st_size > 0

if _has_video(out_mp4):
    print(f'Video already present, skipping render -> {out_mp4}')
else:
    import imageio.v2 as imageio
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    from mpl_toolkits.mplot3d import Axes3D  # noqa (registers '3d')

    SKELETON = [(valid_kps.index(m['moving']), valid_kps.index(r))
                for m in MECHANISMS.values() for r in m['refs']
                if m['moving'] in valid_kps and r in valid_kps]

    allp = points_3d[np.isfinite(points_3d).all(axis=-1)]
    lo, hi = np.percentile(allp, [2, 98], axis=0); pad = (hi - lo) * 0.15; lo, hi = lo - pad, hi + pad

    RENDER_START_S, RENDER_DUR_S, RENDER_STRIDE = 0, int(n_frames / FPS), 5
    fidx = list(range(int(RENDER_START_S*FPS), min(n_frames, int((RENDER_START_S+RENDER_DUR_S)*FPS)), RENDER_STRIDE))

    def tier_color(f, k):
        if single_view_mask[f, k]: return 'tomato'
        if interp_mask[f, k]:      return 'mediumpurple'
        return 'forestgreen' if n_views_used[f, k] >= 3 else 'goldenrod'

    writer = imageio.get_writer(str(out_mp4), fps=max(1, round(FPS / RENDER_STRIDE)), codec='libx264', quality=8)
    fig = Figure(figsize=(7, 6)); FigureCanvasAgg(fig); ax = fig.add_subplot(111, projection='3d')

    for f in fidx:
        ax.clear()
        ax.set_xlim(lo[0], hi[0]); ax.set_ylim(lo[1], hi[1]); ax.set_zlim(lo[2], hi[2])
        ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
        pts = points_3d[f]
        for i, j in SKELETON:
            if np.isfinite(pts[i]).all() and np.isfinite(pts[j]).all():
                ax.plot(*zip(pts[i], pts[j]), color='0.6', lw=1.0)
        for k in range(n_kps):
            if np.isfinite(pts[k]).all():
                ax.scatter(*pts[k], s=30, c=tier_color(f, k), edgecolors='k', lw=0.3)
        ax.set_title(f'{STAGE_NAMES[stage[f]]}  | frame {f} ({f/FPS:.1f}s)', fontsize=10)
        fig.canvas.draw()
        writer.append_data(np.asarray(fig.canvas.buffer_rgba())[..., :3])
    writer.close()
    print(f'3D reconstruction -> {out_mp4.resolve()}  ({len(fidx)} frames)')

Video already present, skipping render -> /home/kenny/HTCV/data/lockbox_state/scene1/lockbox_3d_reconstruction.mp4


## 13. Save `lockbox_state.csv`

Per mechanism: state value, view count, single-view flag, engaged flag (+ `_x/_y/_z` for `free`).
Plus the discrete `stage` / `stage_name`.

In [ ]:
out = {'frame': np.arange(n_frames), 'time_s': np.arange(n_frames) / FPS,
       'stage': stage, 'stage_name': [STAGE_NAMES[s] for s in stage]}
for name, st in state.items():
    mv_idx = valid_kps.index(MECHANISMS[name]['moving'])
    out[name]                  = st['value']
    out[f'{name}_nviews']      = st['nviews']
    out[f'{name}_single_view'] = single_view_mask[:, mv_idx]
    if name in engaged: out[f'{name}_engaged'] = engaged[name]
    if st['kind'] == 'free':
        out[f'{name}_x'] = st['xyz'][:, 0]
        out[f'{name}_y'] = st['xyz'][:, 1]
        out[f'{name}_z'] = st['xyz'][:, 2]

df_state = pd.DataFrame(out)
csv_path = OUT_DIR / 'lockbox_state.csv'
df_state.to_csv(csv_path, index=False)
print(f'Saved -> {csv_path}  ({len(df_state)} frames, {df_state.shape[1]} cols)')
df_state.head()

Saved -> /home/kenny/HTCV/data/lockbox_state/scene1/lockbox_state.csv  (8190 frames, 23 cols)


,frame,time_s,stage,stage_name,lever1,lever1_nviews,lever1_single_view,lever1_engaged,slider1,slider1_nviews,...,ball1_nviews,ball1_single_view,ball1_engaged,ball1_x,ball1_y,ball1_z,cover1,cover1_nviews,cover1_single_view,cover1_engaged
0,0,0.000000,0,start,0.232943,1,False,False,-0.009694,3,...,3,False,False,46.533935,-21.244876,28.448763,-0.101646,2,False,False
1,1,0.033333,0,start,0.232939,1,False,False,-0.009108,3,...,3,False,False,46.537330,-21.288238,28.451606,-0.110204,2,False,False
2,2,0.066667,0,start,0.232933,1,False,False,-0.011973,3,...,3,False,False,46.538577,-21.267027,28.449464,-0.113117,2,False,False
3,3,0.100000,0,start,0.232727,1,False,False,-0.017362,3,...,3,False,False,46.530656,-21.241025,28.456682,-0.035720,2,False,False
4,4,0.133333,0,start,0.232777,1,False,False,-0.013277,3,...,3,False,False,46.532519,-21.254210,28.488784,-0.076138,2,False,False


## 14. Save `lockbox_state.pkl` (features + ground truth for the next notebook)

Holds raw 2D, reconstructed 3D, provenance, continuous states, engagement, and the discrete `stage`
everything the state-prediction notebook needs to compare 2D / 3D / mouse-based predictors against truth.

In [ ]:
state_out = dict(
    points_2d=points_2d, points_3d=points_3d, likelihoods=likelihoods,
    n_views_used=n_views_used, single_view_mask=single_view_mask, interp_mask=interp_mask,
    valid_kps=valid_kps, view_order=view_order, n_frames=n_frames, n_kps=n_kps,
    MECHANISMS=MECHANISMS, STATE_RANGE=STATE_RANGE, RANGE_BOX=RANGE_BOX,
    state={k: dict(v) for k, v in state.items()},
    engaged=engaged, stage=stage, onsets=onsets,
    STAGE_ORDER=STAGE_ORDER, STAGE_NAMES=STAGE_NAMES, ENGAGE=ENGAGE,
    VIDEO_FOR_VIEW={k: str(v) for k, v in VIDEO_FOR_VIEW.items()},
    schema_path=str(SCHEMA_PATH), calibration_source='lockbox_pnp_perview',
)
with open(OUT_DIR / 'lockbox_state.pkl', 'wb') as f:
    pickle.dump(state_out, f)
print(f'State -> {OUT_DIR / "lockbox_state.pkl"}')
print(f'   coverage {np.isfinite(points_3d).all(axis=-1).mean()*100:.1f}%  |  '
      f'reached stage {int(stage.max())}/{len(STAGE_ORDER)} ({STAGE_NAMES[int(stage.max())]})')

State -> /home/kenny/HTCV/data/lockbox_state/scene1/lockbox_state.pkl
   coverage 82.8%  |  reached stage 4/4 (cover_slid)
